In [1]:
import torch
print(torch.__version__)

2.14.1+cpu


In [18]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings("ignore")


In [3]:
DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(exist_ok=True)

posts_df = pd.read_csv(DATA_DIR / "the-reddit-dataset-dataset-posts.csv")
comments_df = pd.read_csv(DATA_DIR / "the-reddit-dataset-dataset-comments.csv")

posts_df = posts_df[posts_df["selftext"].notna()].copy()
comments_df = comments_df[comments_df["body"].notna()].copy()

removed_marks = ["[removed]", "[deleted]", "[ removed by reddit ]", "[deleted by user]"]
posts_df = posts_df[~posts_df["selftext"].str.strip().str.lower().isin(removed_marks)].copy()
comments_df = comments_df[~comments_df["body"].str.strip().str.lower().isin(removed_marks)].copy()

posts_df.shape, comments_df.shape

((10977, 13), (51255, 11))

In [8]:
# 텍스트 정제
def clean_text(t):
    t = str(t).lower()
    t = re.sub(r"http\S+", "", t)
    t = re.sub(r"[^a-zA-Z\s]", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

posts_df["clean_text"] = posts_df["selftext"].apply(clean_text)
comments_df["clean_text"] = comments_df["body"].apply(clean_text)

posts_df = posts_df[posts_df["clean_text"] != ""].copy()
comments_df = comments_df[comments_df["clean_text"] != ""].copy()

posts_df.shape, comments_df.shape

((10939, 14), (50462, 12))

In [9]:
#SentenceTransformer 임베딩
embedder = SentenceTransformer("all-MiniLM-L6-v2")

post_embeddings = embedder.encode(
    posts_df["clean_text"].tolist(),
    show_progress_bar=True
)

comment_embeddings = embedder.encode(
    comments_df["clean_text"].tolist(),
    show_progress_bar=True
)


Batches:   0%|          | 0/342 [00:00<?, ?it/s]

Batches:   0%|          | 0/1577 [00:00<?, ?it/s]

In [ ]:
# # print(posts_df.columns)
# # print(comments_df.columns)

# Index(['index', 'type', 'id', 'subreddit.id', 'subreddit.name',
#        'subreddit.nsfw', 'created_utc', 'permalink', 'domain', 'url',
#        'selftext', 'title', 'score', 'clean_text', 'topic'],
#       dtype='object')
# Index(['index', 'type', 'id', 'subreddit.id', 'subreddit.name',
#        'subreddit.nsfw', 'created_utc', 'permalink', 'body', 'sentiment',
#        'score', 'clean_text'],
#       dtype='object')

In [10]:
# BERTopic 학습 (Posts)
topic_model_posts = BERTopic(verbose=True)

post_topics, _ = topic_model_posts.fit_transform(
    posts_df["clean_text"],
    post_embeddings
)

posts_df["topic"] = post_topics
topic_model_posts.get_topic_info().head()


2026-10-23 12:04:18,074 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-23 12:04:44,894 - BERTopic - Dimensionality - Completed ✓
2026-10-23 12:04:44,898 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-23 12:04:45,324 - BERTopic - Cluster - Completed ✓
2026-10-23 12:04:45,333 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-23 12:04:45,804 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,4027,-1_to_the_for_of,"[to, the, for, of, and, data, that, is, in, on]",[hi if it is not a problem i would like to ask...
1,0,211,0_comments_reddit_submissions_comment,"[comments, reddit, submissions, comment, subre...",[endpoint there is a lot you can do with this ...
2,1,205,1_election_political_elections_votes,"[election, political, elections, votes, vote, ...",[hi all i have searched high and low for a pre...
3,2,188,2_product_customer_sales_products,"[product, customer, sales, products, retail, p...",[does anyone have a dataset which has a substa...
4,3,177,3_covid_cases_gt_coronavirus,"[covid, cases, gt, coronavirus, virus, hopkins...",[you have probably seen most of these but i th...


In [11]:
# BERTopic 학습 (Comments)
topic_model_comments = BERTopic(verbose=True)

comment_topics, _ = topic_model_comments.fit_transform(
    comments_df["clean_text"],
    comment_embeddings
)

comments_df["topic"] = comment_topics
topic_model_comments.get_topic_info().head()


2026-10-23 12:04:46,459 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


2026-10-23 12:05:03,866 - BERTopic - Dimensionality - Completed ✓
2026-10-23 12:05:03,868 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-23 12:05:11,330 - BERTopic - Cluster - Completed ✓
2026-10-23 12:05:11,346 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-23 12:05:12,720 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,20289,-1_amp_the_it_to,"[amp, the, it, to, data, of, that, in, is, and]",[wow well we should talk about what a query re...
1,0,1011,0_player_games_game_football,"[player, games, game, football, stats, players...","[soccer games or other sports, nfl player game..."
2,1,846,1_data_world_kind_what,"[data, world, kind, what, thanks, looking, int...",[can t help you on the data but i wanted to sa...
3,2,503,2_twitter_tweets_tweet_api,"[twitter, tweets, tweet, api, ids, firehose, t...",[so basically i can download tweets i cannot l...
4,3,429,3_images_image_pose_estimation,"[images, image, pose, estimation, object, boun...",[i was looking for the images do you know wher...


In [12]:
# Noise(-1) 제거
posts_df_valid = posts_df[posts_df["topic"] != -1].copy()
comments_df_valid = comments_df[comments_df["topic"] != -1].copy()

posts_df_valid.shape, comments_df_valid.shape


((6912, 15), (30173, 13))

In [13]:
TOP_N = 10

top_topics = (
    posts_df_valid["topic"]
    .value_counts()
    .head(TOP_N)
    .index
)

posts_top = posts_df_valid[posts_df_valid["topic"].isin(top_topics)]
comments_top = comments_df_valid[comments_df_valid["topic"].isin(top_topics)]


In [14]:
# Post vs Comment 토픽 분포 비교
post_dist = posts_top["topic"].value_counts(normalize=True)
comment_dist = comments_top["topic"].value_counts(normalize=True)

topic_compare = pd.concat(
    [
        post_dist.rename("post_ratio"),
        comment_dist.rename("comment_ratio")
    ],
    axis=1
).fillna(0)

topic_compare


,post_ratio,comment_ratio
topic,,
0,0.127415,0.214969
1,0.123792,0.179885
2,0.113527,0.106953
3,0.106884,0.091218
4,0.100845,0.077823
5,0.095411,0.069105
6,0.089372,0.068467
7,0.087560,0.068042
8,0.082126,0.062939


In [15]:
# Subreddit별 대표 토픽
posts_sub = posts_df_valid.rename(columns={"subreddit.name": "subreddit"})
posts_sub = posts_sub[posts_sub["subreddit"].notna()]

subreddit_top_topic = (
    posts_sub.groupby("subreddit")["topic"]
    .value_counts()
    .groupby(level=0)
    .head(1)
    .reset_index(name="count")
)

subreddit_top_topic.head()


,subreddit,topic,count
0,datasets,0,211


In [16]:
def get_topic_keywords(model, topic_id, n=10):
    return ", ".join([w for w, _ in model.get_topic(topic_id)[:n]])

summary_rows = []

for topic_id in top_topics:
    summary_rows.append({
        "topic": topic_id,
        "keywords": get_topic_keywords(topic_model_posts, topic_id),
        "examples": (
            posts_df_valid[posts_df_valid["topic"] == topic_id]
            .head(3)["clean_text"]
            .tolist()
        )
    })

topic_summary_df = pd.DataFrame(summary_rows)
topic_summary_df


,topic,keywords,examples
0,0,"comments, reddit, submissions, comment, subred...",[cid qmtyhedecjrteyp rsztugilsewnj c ciwtjecwz...
1,1,"election, political, elections, votes, vote, v...",[want to see detail info of how political peop...
2,2,"product, customer, sales, products, retail, pr...",[hi can anyone help me get the dataset of chip...
3,3,"covid, cases, gt, coronavirus, virus, hopkins,...",[a quick google search of party breakdown of c...
4,4,"images, image, object, dataset, detection, bou...",[i want to classify if an image contains a con...
5,5,"weather, temperature, hourly, precipitation, c...",[i m working on a project and am looking for a...
6,6,"crime, police, shootings, crimes, gun, mass, v...",[hey guys checkout this new dataset on various...
7,7,"variables, regression, variable, least, catego...",[i m working on a stats project to test some o...
8,8,"schools, school, students, education, universi...",[hey everyone i am experiencing quite a diffic...
9,9,"ftp, housing, zillow, estate, property, home, ...",[i have been playing around with zillow s publ...


In [17]:
posts_df_valid.to_csv(
    OUTPUT_DIR / "posts_with_topics.csv",
    index=False
)

comments_df_valid.to_csv(
    OUTPUT_DIR / "comments_with_topics.csv",
    index=False
)

topic_compare.to_csv(
    OUTPUT_DIR / "post_comment_topic_distribution.csv"
)

subreddit_top_topic.to_csv(
    OUTPUT_DIR / "subreddit_top_topics.csv",
    index=False
)

topic_summary_df.to_csv(
    OUTPUT_DIR / "topic_summary_table.csv",
    index=False
)


In [20]:
ids = [0, 13, 21, 41, 10, 2, 20, 7, 35]
names = topic_model_posts.get_topic_info()[["Topic", "Count", "Name"]]

names[["Topic", "Name"]].to_csv(OUTPUT_DIR / "topic_names.csv", index=False)

In [21]:
names[names["Topic"].isin(ids)]

,Topic,Count,Name
1,0,211,0_comments_reddit_submissions_comment
3,2,188,2_product_customer_sales_products
8,7,145,7_variables_regression_variable_least
11,10,107,10_census_population_us_block
14,13,99,13_rep_tweets_twitter_tweet
21,20,82,20_faces_face_facial_images
22,21,80,21_we_our_your_you
36,35,52,35_countries_europe_catholic_country
42,41,47,41_suggestions_complain_visualisations_discuss
